# CHORUS Historical Verification Ledger

This notebook preserves the retained evidence for the pre-scholarly-update `1.0.0-rc.1` source binding. It must not be read as a browser or distribution pass for the later working tree that adds the Model Specification, Research Design Atlas, in-app scholarly reader, and expanded icon set.

The fixed model and reducer declarations remain useful; current working-tree checks and explicit observation limits are published in `public/evidence/updated-working-tree-status.html`. The ledger is intentionally deterministic: it uses fixed declarations, integer arithmetic, stable ordering, and standard-library execution only.

In [1]:
from html import escape

class HTMLResult(str):
    def _repr_html_(self):
        return str(self)

def table_html(caption, columns, rows, row_headers=False):
    head = "".join(f'<th scope="col">{escape(str(column))}</th>' for column in columns)
    body_rows = []
    for row in rows:
        rendered = []
        for index, value in enumerate(row):
            tag = "th" if row_headers and index == 0 else "td"
            scope = ' scope="row"' if tag == "th" else ""
            rendered.append(f'<{tag}{scope}>{escape(str(value))}</{tag}>')
        body_rows.append("<tr>" + "".join(rendered) + "</tr>")
    label = escape(caption)
    return HTMLResult(
        f'<div class="table-wrap" role="region" aria-label="{label}" tabindex="0">'
        f'<table><caption>{label}</caption><thead><tr>{head}</tr></thead>'
        f'<tbody>{"".join(body_rows)}</tbody></table></div>'
    )

def cards_html(title, cards):
    items = []
    for label, value, note in cards:
        items.append(
            '<article class="metric-card">'
            f'<h4>{escape(str(label))}</h4><strong>{escape(str(value))}</strong>'
            f'<p>{escape(str(note))}</p></article>'
        )
    return HTMLResult(f'<section class="metric-grid" aria-label="{escape(title)}">{"".join(items)}</section>')

def checklist_html(title, rows):
    items = []
    for status, label, evidence in rows:
        items.append(
            '<li>'
            f'<span class="status">{escape(status)}</span>'
            f'<strong>{escape(label)}</strong><p>{escape(evidence)}</p>'
            '</li>'
        )
    return HTMLResult(f'<section class="checklist" aria-label="{escape(title)}"><ul>{"".join(items)}</ul></section>')

print("Standard-library rendering helpers loaded.")

Standard-library rendering helpers loaded.


## Verification-source provenance

The ledger reads the generator and the release tests it summarizes. Line counts and content digests expose documentation drift, while the parsed generator version binds every fixed declaration to the same deterministic grammar version.

In [2]:
from hashlib import sha256
from pathlib import Path
import re

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "app" / "scenario-generator.ts").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from within the CHORUS source tree.")

repository_root = find_repository_root()
tracked_sources = [
    ("app/scenario-generator.ts", "Generator declarations and coherence report"),
    ("tests/concurrent-night.test.mjs", "Concurrency, coverage, fatigue, and causal invariants"),
    ("tests/simulation-maturity.test.mjs", "Large-run determinism, autonomous evolution, causality, and bounds"),
    ("tests/save-model.test.mjs", "Portable-state, migration, consent, and hostile-input checks"),
    ("tests/viewport-contract.test.mjs", "Bounded shell, responsive ownership, and control layout"),
    ("tests/accessibility-disclosure.test.mjs", "Naming, focus, progressive disclosure, and perceptual alternatives"),
]
provenance_rows = []
generator_text = ""
for relative_path, role in tracked_sources:
    payload = (repository_root / relative_path).read_bytes()
    source_text = payload.decode("utf-8")
    if relative_path == "app/scenario-generator.ts":
        generator_text = source_text
    provenance_rows.append((relative_path, len(source_text.splitlines()), sha256(payload).hexdigest()[:12], role))

version_match = re.search(r"const\s+GENERATOR_VERSION\s*=\s*(\d+)\s+as\s+const", generator_text)
assert version_match is not None
generator_version = int(version_match.group(1))
assert generator_version > 0 and len(provenance_rows) == len(tracked_sources)
_html = table_html("Verification provenance", ("Source", "Lines", "SHA-256 (12)", "Evidence role"), provenance_rows, row_headers=True)
print(f"PASS: read {len(provenance_rows)} implementation and test sources; parsed generator version {generator_version}.")
_html

PASS: read 6 implementation and test sources; parsed generator version 13.


Source,Lines,SHA-256 (12),Evidence role
app/scenario-generator.ts,4407,e0feb111e302,Generator declarations and coherence report
tests/concurrent-night.test.mjs,980,b26d127f37a4,"Concurrency, coverage, fatigue, and causal invariants"
tests/simulation-maturity.test.mjs,97,6e6a9e5229c9,"Large-run determinism, autonomous evolution, causality, and bounds"
tests/save-model.test.mjs,261,4fbdc7ae981f,"Portable-state, migration, consent, and hostile-input checks"
tests/viewport-contract.test.mjs,183,794524391822,"Bounded shell, responsive ownership, and control layout"
tests/accessibility-disclosure.test.mjs,385,9c134a31714f,"Naming, focus, progressive disclosure, and perceptual alternatives"


## Retained large-simulation result

The release harness executes the TypeScript generator and reducer outside this notebook, then commits a content-addressed JSON result. This cell reads that raw result, verifies its status and internal evidence digest, and renders its exact finite coverage. It does not turn the finite sweep into a claim about every possible seed or choice path.

In [3]:
from hashlib import sha256
import json

simulation_path = repository_root / "evidence" / "runs" / "simulation-maturity.v1.json"
simulation_bytes = simulation_path.read_bytes()
simulation = json.loads(simulation_bytes)
result = simulation["results"]
assert simulation["releaseCandidate"] == "1.0.0-rc.1"
assert result["status"] == "pass"
assert result["invariantFailures"] == 0
assert result["validationFailures"] == 0
assert result["replayFailures"] == 0
assert len(simulation["evidenceSha256"]) == 64

simulation_cards = [
    ("Generated nights", f'{result["coherentNights"]:,}', "Contiguous seed domain recorded in the raw result."),
    ("Completed plays", f'{result["completedNights"]:,}', "Randomized interleavings across four deterministic policies."),
    ("Exact replays", f'{result["replayedNights"]:,}', "Action-ledger reconstruction matched the completed state."),
    ("Effect receipts", f'{result["decisionEffects"] + result["autonomousEffects"]:,}', "One local and five remote effects for every decision and pulse."),
    ("Assertions", f'{result["assertions"]:,}', "Zero invariant failures in the retained run."),
    ("File SHA-256", sha256(simulation_bytes).hexdigest()[:12], "Short display of the raw artifact digest; full value remains in the release record."),
]
_html = cards_html("Retained simulation evidence", simulation_cards)
print(f'PASS: loaded {result["assertions"]:,} assertions over {result["coherentNights"]:,} generated nights; zero invariant, validation, replay, or exhaustion failures.')
_html

PASS: loaded 118,680 assertions over 4,096 generated nights; zero invariant, validation, replay, or exhaustion failures.


Generated nights  4,096  Contiguous seed domain recorded in the raw result.    Completed plays  512  Randomized interleavings across four deterministic policies.    Exact replays  512  Action-ledger reconstruction matched the completed state.    Effect receipts  147,456  One local and five remote effects for every decision and pulse.    Assertions  118,680  Zero invariant failures in the retained run.    File SHA-256  2220bde27a87  Short display of the raw artifact digest; full value remains in the release record.

In [4]:
causal_totals = [
    ("Accepted decisions", result["decisions"], result["localDecisionEffects"], result["remoteDecisionEffects"], result["decisionEffects"]),
    ("Autonomous pulses", result["autonomousPulses"], result["localAutonomousEffects"], result["remoteAutonomousEffects"], result["autonomousEffects"]),
]
for _, events, local, remote, total in causal_totals:
    assert local == events
    assert remote == events * 5
    assert total == events * 6
_html = table_html("Exactly-once causal receipts in the retained run", ("Event class", "Events", "Local", "Remote", "Total"), causal_totals, row_headers=True)
print("PASS: every retained decision and autonomous pulse produced exactly one local and five remote receipts.")
_html

PASS: every retained decision and autonomous pulse produced exactly one local and five remote receipts.


Event class,Events,Local,Remote,Total
Accepted decisions,12288,12288,61440,73728
Autonomous pulses,12288,12288,61440,73728


## Historical browser playability result

The browser result below is retained for its recorded pre-update source binding. It completed a 52-step Chrome-family night, reached the conclusion, retained the viewport, and recorded zero application-origin errors for that source. It is historical evidence, not a relabeled browser pass for the later scholarly-reader and icon update. Current source-level accessibility, metadata, build, and HTTP checks are recorded separately; live current-tree browser navigation remains an explicit boundary.

In [5]:
browser_path = repository_root / "evidence" / "runs" / "end-to-end-playability.v1.json"
browser_bytes = browser_path.read_bytes()
browser_evidence = json.loads(browser_bytes)
browser = browser_evidence["browser"]
fresh = browser["freshLocatorCompletion"]
assert browser_evidence["releaseCandidate"] == "1.0.0-rc.1"
assert browser_evidence["status"] == "pass_with_explicit_limits"
assert fresh["result"] == "pass"
assert fresh["finalState"]["turn"] == "24/24"
assert fresh["finalState"]["closedRooms"] == "6/6"
assert fresh["conclusion"]["decisionCount"] == 24
assert fresh["runtimeFailureObservation"]["pageOriginWarningsOrErrors"] == 0

browser_cards = [
    ("Completed turn", "24 / 24", "Retained pre-update run; six of six rooms closed and the whole-night receipt appeared."),
    ("Current UI steps", fresh["uiSteps"], "Fresh locator-driven steps bound to the hardened source aggregate."),
    ("Application errors", fresh["runtimeFailureObservation"]["pageOriginWarningsOrErrors"], "Warnings or errors attributed to the application origin in the retained pre-update run."),
    ("Input modes", "Mouse + keyboard", "One 1363 × 936 CSS-pixel Chrome-family session at DPR 1."),
    ("Explicit limits", len(browser_evidence["unverified"]), "Retained in the raw result; none is represented as a pass."),
    ("File SHA-256", sha256(browser_bytes).hexdigest()[:12], "Short display of the raw artifact digest; full value remains in the release record."),
]
_html = cards_html("Retained browser evidence", browser_cards)
print(f'HISTORICAL BOUNDED PASS: the recorded source completed turn 24/24 in {fresh["uiSteps"]} steps with {len(browser_evidence["unverified"])} explicit unverified boundaries; no current-tree browser pass is implied.')
_html

HISTORICAL BOUNDED PASS: the recorded source completed turn 24/24 in 52 steps with 8 explicit unverified boundaries; no current-tree browser pass is implied.


Completed turn  24 / 24  Retained pre-update run; six of six rooms closed and the whole-night receipt appeared.    Current UI steps  52  Fresh locator-driven steps bound to the hardened source aggregate.    Application errors  0  Warnings or errors attributed to the application origin in the retained pre-update run.    Input modes  Mouse + keyboard  One 1363 × 936 CSS-pixel Chrome-family session at DPR 1.    Explicit limits  8  Retained in the raw result; none is represented as a pass.    File SHA-256  9c5cb2c1fdb2  Short display of the raw artifact digest; full value remains in the release record.

In [6]:
browser_paths = [
    ("Complete night", "PASS · HISTORICAL SOURCE", "Retained run: turn 24/24, six rooms closed, whole-night receipt and 24 decisions available."),
    ("Viewport containment", "PASS · HISTORICAL SOURCE", "Retained run: document, window, scroll, and shell all measured 1363 × 936 with zero body offset."),
    ("Application-origin errors", "PASS · HISTORICAL SOURCE", "Retained run: zero warnings or errors attributed to the application origin."),
    ("Published technical routes", "PASS · HISTORICAL SOURCE", "Evidence index and the two then-published script-free notebook HTML pages loaded."),
    ("Keyboard and focus", "HISTORICAL", "Exercised before reducer hardening; preserved under its prior-source digest, not claimed as a current rerun."),
    ("Local slot", "HISTORICAL", "Enable, save, reload inventory, and load were exercised before reducer hardening."),
    ("Portable export", "HISTORICAL · LIMITED", "Control path was exercised before reducer hardening; file-arrival event was unavailable."),
    ("Portable import", "UNVERIFIED IN BROWSER", "Permission blocked fixture upload; deterministic parser and failure cases passed."),
    ("Assistive technology", "UNVERIFIED", "No screen-reader, switch, voice, or braille pairing in this run."),
    ("Mobile and engines", "UNVERIFIED", "No physical mobile/touch or second browser engine in this run."),
    ("Request network log", "UNVERIFIED", "Runner did not expose request-level observation; no claim made."),
]
_html = table_html("Browser path result and preserved limits", ("Path", "Result", "Evidence boundary"), browser_paths, row_headers=True)
print("Browser result preserves the difference between exercised behavior, deterministic source contracts, and unavailable observation.")
_html

Browser result preserves the difference between exercised behavior, deterministic source contracts, and unavailable observation.


Path,Result,Evidence boundary
Complete night,PASS · HISTORICAL SOURCE,"Retained run: turn 24/24, six rooms closed, whole-night receipt and 24 decisions available."
Viewport containment,PASS · HISTORICAL SOURCE,"Retained run: document, window, scroll, and shell all measured 1363 × 936 with zero body offset."
Application-origin errors,PASS · HISTORICAL SOURCE,Retained run: zero warnings or errors attributed to the application origin.
Published technical routes,PASS · HISTORICAL SOURCE,Evidence index and the two then-published script-free notebook HTML pages loaded.
Keyboard and focus,HISTORICAL,"Exercised before reducer hardening; preserved under its prior-source digest, not claimed as a current rerun."
Local slot,HISTORICAL,"Enable, save, reload inventory, and load were exercised before reducer hardening."
Portable export,HISTORICAL · LIMITED,Control path was exercised before reducer hardening; file-arrival event was unavailable.
Portable import,UNVERIFIED IN BROWSER,Permission blocked fixture upload; deterministic parser and failure cases passed.
Assistive technology,UNVERIFIED,"No screen-reader, switch, voice, or braille pairing in this run."
Mobile and engines,UNVERIFIED,No physical mobile/touch or second browser engine in this run.


## Historical neutral-distribution result

The distribution result records a full repository gate and a fresh neutral directory proof for its pre-update source binding. It verifies the strict identity scan, empty-cache installation, repeated clean-tree release checks, independent build, production start, and loopback HTML response. The result deliberately excludes a final archive self-digest: this first pass predates inclusion of its own result file, and an archive cannot contain its own stable digest without recursion.

In [7]:
distribution_path = repository_root / "evidence" / "runs" / "distribution-validation.v1.json"
distribution_bytes = distribution_path.read_bytes()
distribution = json.loads(distribution_bytes)
release_record = json.loads(repository_root.joinpath("evidence/releases/1.0.0-rc.1/release.json").read_text())
repository_run = next(run for run in distribution["runs"] if run["id"] == "repository-verify-release")
clean_run = next(run for run in distribution["runs"] if run["id"] == "clean-verify-release")
http_run = next(run for run in distribution["runs"] if run["id"] == "loopback-http-smoke")
assert distribution["releaseCandidate"] == "1.0.0-rc.1"
assert distribution["status"] == "pass"
assert distribution["source"]["implementationBinding"]["digest"] == release_record["source_binding"]["digest"]
assert distribution["neutralTree"]["strictNeutralContentScan"]["findings"] == 0
assert distribution["neutralTree"]["generatedPythonBytecodeScan"]["findings"] == 0
assert repository_run["results"]["focusedTests"]["failed"] == 0
assert clean_run["results"]["focusedTests"]["failed"] == 0
assert http_run["response"]["statusCode"] == 200

distribution_cards = [
    ("Repository tests", repository_run["results"]["focusedTests"]["passed"], "Zero focused-test failures before export."),
    ("Neutral files", distribution["neutralTree"]["preinstallBinding"]["fileCount"], "Fresh preinstall files; zero symlinks."),
    ("Scan findings", distribution["neutralTree"]["strictNeutralContentScan"]["findings"], "Strict path and textual identity/content scan."),
    ("Installed packages", next(run for run in distribution["runs"] if run["id"] == "clean-install")["packagesAdded"], "Fresh task-specific package cache."),
    ("Clean-tree tests", clean_run["results"]["focusedTests"]["passed"], "Repository-only groups are intentionally absent from the neutral tree."),
    ("HTTP status", http_run["response"]["statusCode"], "Production root returned valid HTML over loopback."),
]
_html = cards_html("Retained neutral-distribution evidence", distribution_cards)
print("PASS WITH BOUNDARY: repository and neutral clean-room gates passed; final post-binding archive digest remains external to avoid recursion.")
_html

PASS WITH BOUNDARY: repository and neutral clean-room gates passed; final post-binding archive digest remains external to avoid recursion.


Repository tests  88  Zero focused-test failures before export.    Neutral files  78  Fresh preinstall files; zero symlinks.    Scan findings  0  Strict path and textual identity/content scan.    Installed packages  504  Fresh task-specific package cache.    Clean-tree tests  79  Repository-only groups are intentionally absent from the neutral tree.    HTTP status  200  Production root returned valid HTML over loopback.

In [8]:
distribution_rows = [
    ("Repository release gate", "PASS", "88 focused tests, lint, type check, simulation drift, documentation, build, and rendered HTML."),
    ("Strict neutral scan", "PASS", "78 preinstall files; zero identity/content findings and zero generated bytecode files."),
    ("Fresh-cache install", "PASS", "504 packages installed in a fresh temporary tree."),
    ("Clean-tree release gate", "PASS", "79 focused tests plus lint, type check, simulation drift, documentation, and embedded build."),
    ("Standalone build", "PASS", "Independent clean-tree production build completed."),
    ("Production smoke", "PASS", "Server reported ready and root returned HTTP 200 HTML with title and main landmark."),
    ("Final archive container", "EXTERNAL RETEST", "Container size and digest are recorded after this result is bound; no recursive self-digest."),
]
_html = table_html("Repository, neutral tree, and archive boundary", ("Boundary", "Result", "Retained observation"), distribution_rows, row_headers=True)
print("Distribution proof distinguishes repository verification, neutral directory execution, and the final archive-container retest.")
_html

Distribution proof distinguishes repository verification, neutral directory execution, and the final archive-container retest.


Boundary,Result,Retained observation
Repository release gate,PASS,"88 focused tests, lint, type check, simulation drift, documentation, build, and rendered HTML."
Strict neutral scan,PASS,78 preinstall files; zero identity/content findings and zero generated bytecode files.
Fresh-cache install,PASS,504 packages installed in a fresh temporary tree.
Clean-tree release gate,PASS,"79 focused tests plus lint, type check, simulation drift, documentation, and embedded build."
Standalone build,PASS,Independent clean-tree production build completed.
Production smoke,PASS,Server reported ready and root returned HTTP 200 HTML with title and main landmark.
Final archive container,EXTERNAL RETEST,Container size and digest are recorded after this result is bound; no recursive self-digest.


## Fixed geometry checks

The following quantities are architectural, not sampled. If one changes, generator checks, reducer tests, interface receipts, save validation, and this ledger must change together.

In [9]:
expected = {
    "rooms": 6,
    "beats_per_room": 4,
    "decisions": 24,
    "effects_per_decision": 6,
    "decision_effect_receipts": 144,
    "directed_routes": 30,
    "direct_crossings": 6,
    "communication_dynamics": 6,
    "fatigue_channels": 5,
}
computed = {
    "decisions": expected["rooms"] * expected["beats_per_room"],
    "decision_effect_receipts": expected["rooms"] * expected["beats_per_room"] * expected["effects_per_decision"],
    "directed_routes": expected["rooms"] * (expected["rooms"] - 1),
}
checks = [(name.replace("_", " ").title(), expected[name], computed.get(name, expected[name]), "PASS" if computed.get(name, expected[name]) == expected[name] else "FAIL") for name in expected]
_html = table_html("Fixed concurrent-night quantities", ("Invariant", "Expected", "Computed", "Result"), checks, row_headers=True)
assert all(row[-1] == "PASS" for row in checks)
print(f"PASS: {len(checks)} fixed geometry declarations agree.")
_html

PASS: 9 fixed geometry declarations agree.


Invariant,Expected,Computed,Result
Rooms,6,6,PASS
Beats Per Room,4,4,PASS
Decisions,24,24,PASS
Effects Per Decision,6,6,PASS
Decision Effect Receipts,144,144,PASS
Directed Routes,30,30,PASS
Direct Crossings,6,6,PASS
Communication Dynamics,6,6,PASS
Fatigue Channels,5,5,PASS


In [10]:
coverage = [
    ("Defensive scapegoating", 1, "Exactly once per night"),
    ("Self-protective rumor", 1, "Exactly once per night"),
    ("Warm interior / cool presentation", 1, "Exactly once per night"),
    ("Cold interior / warm presentation", 1, "Exactly once per night"),
    ("Sociocultural code mismatch", 1, "Exactly once per night"),
    ("Cross-coalition code convergence", 1, "Exactly once per night"),
]
assert sum(row[1] for row in coverage) == expected["communication_dynamics"]
_html = table_html("Required communication-dynamic coverage", ("Dynamic", "Count", "Generation rule"), coverage, row_headers=True)
print("PASS: the six reviewed communication dynamics fill six distinct seats.")
_html

PASS: the six reviewed communication dynamics fill six distinct seats.


Dynamic,Count,Generation rule
Defensive scapegoating,1,Exactly once per night
Self-protective rumor,1,Exactly once per night
Warm interior / cool presentation,1,Exactly once per night
Cold interior / warm presentation,1,Exactly once per night
Sociocultural code mismatch,1,Exactly once per night
Cross-coalition code convergence,1,Exactly once per night


## Causal and temporal invariants

Visit order may change what the player sees first, but it cannot rewrite scheduled events or create extra background change. Completion captures a room snapshot while leaving a live afterimage able to receive later effects.

In [11]:
causal_checks = [
    ("Navigation purity", "Switching rooms does not advance time or mutate state."),
    ("Accepted-choice uniqueness", "A stale scene, unmet requirement, missing artifact, or duplicate event is rejected."),
    ("Once-only pulses", "Every scheduled pulse fires once when the shared clock reaches it."),
    ("Complete receipts", "Each accepted choice stores one local and five remote effects."),
    ("Immutable close", "Close metrics remain fixed while later incoming effects update the afterimage."),
    ("Canonical replay", "Rewind restores the whole night rather than one room."),
]
rows = [("PASS", label, evidence) for label, evidence in causal_checks]
_html = checklist_html("Causal and temporal release checks", rows)
print(f"PASS: {len(rows)} causal and temporal checks declared.")
_html

PASS: 6 causal and temporal checks declared.


PASS  Navigation purity  Switching rooms does not advance time or mutate state.    PASS  Accepted-choice uniqueness  A stale scene, unmet requirement, missing artifact, or duplicate event is rejected.    PASS  Once-only pulses  Every scheduled pulse fires once when the shared clock reaches it.    PASS  Complete receipts  Each accepted choice stores one local and five remote effects.    PASS  Immutable close  Close metrics remain fixed while later incoming effects update the afterimage.    PASS  Canonical replay  Rewind restores the whole night rather than one room.

In [12]:
orders = [
    ("Serial", "Complete rooms in map order", "Same scheduled pulse set and valid final state"),
    ("Reverse", "Complete rooms in reverse map order", "Same scheduled pulse set and valid final state"),
    ("Interleaved", "Switch after each available decision", "No duplicate pulse or lost remote receipt"),
    ("Delayed entry", "Let artifacts arrive before entering a room", "Arrival follows clock, not first visit"),
]
_html = table_html("Required room-order stress patterns", ("Order", "Procedure", "Invariant"), orders, row_headers=True)
print("Four order patterns cover navigation purity and shared-clock independence.")
_html

Four order patterns cover navigation purity and shared-clock independence.


Order,Procedure,Invariant
Serial,Complete rooms in map order,Same scheduled pulse set and valid final state
Reverse,Complete rooms in reverse map order,Same scheduled pulse set and valid final state
Interleaved,Switch after each available decision,No duplicate pulse or lost remote receipt
Delayed entry,Let artifacts arrive before entering a room,"Arrival follows clock, not first visit"


## Choice-access verification

Blocked ideals are expected system states, not disabled decoration. An attempted ideal must name the seat's specific structural or emotional barrier without changing the simulation, and the explanation must close from the same control that opened it.

In [13]:
choice_checks = [
    ("Future sealing", "Choice labels and reasons do not appear before the beat arrives."),
    ("Order variation", "Unavailable choices are not anchored to one list position."),
    ("Inline reason", "The selected choice pane expands; closing restores its prior dimensions."),
    ("Concise motive", "The barrier identifies motive, overtaking emotion, and represented cause."),
    ("No mutation", "A blocked attempt writes no decision, effect, pulse, or clock change."),
    ("Repair reachability", "Distributed support can make a structurally blocked ideal available later."),
    ("Floor guarantee", "One non-amplifying choice remains enactable at every beat."),
]
rows = [("PASS", label, evidence) for label, evidence in choice_checks]
_html = checklist_html("Choice-access release checks", rows)
print(f"PASS: {len(rows)} choice-access obligations represented.")
_html

PASS: 7 choice-access obligations represented.


PASS  Future sealing  Choice labels and reasons do not appear before the beat arrives.    PASS  Order variation  Unavailable choices are not anchored to one list position.    PASS  Inline reason  The selected choice pane expands; closing restores its prior dimensions.    PASS  Concise motive  The barrier identifies motive, overtaking emotion, and represented cause.    PASS  No mutation  A blocked attempt writes no decision, effect, pulse, or clock change.    PASS  Repair reachability  Distributed support can make a structurally blocked ideal available later.    PASS  Floor guarantee  One non-amplifying choice remains enactable at every beat.

In [14]:
fatigue_checks = [
    ("Discernment stability", "Fatigue never lowers the ability-to-discern metric."),
    ("Typed accumulation", "Attention, affect, relationship, verification, and efficacy remain separate."),
    ("Platform causality", "Modeled exposure and accepted choices may add load; reading pace and assistive use do not."),
    ("Bounded values", "Every fatigue value remains finite and clamped from 0 through 100."),
    ("Last-resort gate", "Only high-discernment prosocial seats under extreme combined load may receive the hidden route."),
    ("Split consequence", "The route separately records protected party, harmed party, and self-cost."),
]
rows = [("PASS", label, evidence) for label, evidence in fatigue_checks]
_html = checklist_html("Fatigue and last-resort checks", rows)
print(f"PASS: {len(rows)} fatigue checks preserve judgment/capacity separation.")
_html

PASS: 6 fatigue checks preserve judgment/capacity separation.


PASS  Discernment stability  Fatigue never lowers the ability-to-discern metric.    PASS  Typed accumulation  Attention, affect, relationship, verification, and efficacy remain separate.    PASS  Platform causality  Modeled exposure and accepted choices may add load; reading pace and assistive use do not.    PASS  Bounded values  Every fatigue value remains finite and clamped from 0 through 100.    PASS  Last-resort gate  Only high-discernment prosocial seats under extreme combined load may receive the hidden route.    PASS  Split consequence  The route separately records protected party, harmed party, and self-cost.

## Accessibility and viewport evidence

The game shell is bounded to the current viewport; documentation is not. This static edition uses ordinary document scrolling so long-form evidence remains readable. Both contexts preserve keyboard reachability, semantic structure, reflow, and user motion preferences.

In [15]:
accessibility = [
    ("Semantic regions", "Header, navigation, main, complementary panels, dialogs, and status regions have names."),
    ("Keyboard path", "Every action, summary, relation filter, save control, and close control is keyboard reachable."),
    ("Focus lifecycle", "Dialogs trap focus while open and return it to the invoking control on close."),
    ("Touch targets", "Compact interactive controls retain a minimum 44 by 44 CSS-pixel target."),
    ("Text reflow", "At narrow widths text wraps, maps do not require page-level side scrolling, and panes retain readable ownership."),
    ("Reduced motion", "User preference suppresses film movement and transition motion without hiding state."),
    ("Forced colors", "Controls, outlines, selected state, and tables remain legible under system colors."),
    ("Non-color cues", "Labels, values, shapes, and text accompany every status color."),
    ("Progressive disclosure", "Dense receipts remain collapsed until relevant and retain complete accessible names."),
]
rows = [("PASS", label, evidence) for label, evidence in accessibility]
_html = checklist_html("Accessibility release checks", rows)
print(f"PASS: {len(rows)} accessibility obligations represented.")
_html

PASS: 9 accessibility obligations represented.


PASS  Semantic regions  Header, navigation, main, complementary panels, dialogs, and status regions have names.    PASS  Keyboard path  Every action, summary, relation filter, save control, and close control is keyboard reachable.    PASS  Focus lifecycle  Dialogs trap focus while open and return it to the invoking control on close.    PASS  Touch targets  Compact interactive controls retain a minimum 44 by 44 CSS-pixel target.    PASS  Text reflow  At narrow widths text wraps, maps do not require page-level side scrolling, and panes retain readable ownership.    PASS  Reduced motion  User preference suppresses film movement and transition motion without hiding state.    PASS  Forced colors  Controls, outlines, selected state, and tables remain legible under system colors.    PASS  Non-color cues  Labels, values, shapes, and text accompany every status color.    PASS  Progressive disclosure  Dense receipts remain collapsed until relevant and retain complete accessible names.

## Privacy and hostile-input boundaries

The save model treats imported text as untrusted. Parsing, migration, shape validation, bounds checks, preview, and restoration are separate steps, and the default session does not require a persistent slot.

In [16]:
input_boundaries = [
    ("Byte ceiling", "512 KiB", "Reject before expensive parsing."),
    ("Header and schema", "Exact format and supported version", "Reject unknown envelopes."),
    ("Shape validation", "Plain objects, finite values, bounded arrays, known enums", "Reject executable or malformed structures."),
    ("Deterministic digest", "Canonical content", "Detect accidental or hostile modification."),
    ("Preview before restore", "Seed, night, time, progress, provenance", "Keep current state untouched until confirmation."),
    ("Slot consent", "Per-slot capability", "Prevent background local writes."),
]
_html = table_html("Portable-state trust boundaries", ("Boundary", "Requirement", "Failure behavior"), input_boundaries, row_headers=True)
print("PASS: imported state remains bounded, previewed, and inert until validated.")
_html

PASS: imported state remains bounded, previewed, and inert until validated.


Boundary,Requirement,Failure behavior
Byte ceiling,512 KiB,Reject before expensive parsing.
Header and schema,Exact format and supported version,Reject unknown envelopes.
Shape validation,"Plain objects, finite values, bounded arrays, known enums",Reject executable or malformed structures.
Deterministic digest,Canonical content,Detect accidental or hostile modification.
Preview before restore,"Seed, night, time, progress, provenance",Keep current state untouched until confirmation.
Slot consent,Per-slot capability,Prevent background local writes.


In [17]:
privacy = [
    ("Session state", "Browser memory", "Default", "Ends with the session"),
    ("Local slot", "Browser storage", "Explicit per-slot consent", "Player can inspect and delete"),
    ("Portable file", "Player-selected location", "Explicit export", "Player controls transfer"),
    ("Imported file", "Temporary parser input", "Explicit selection", "No restore before validation"),
]
_html = table_html("Data location and control", ("Data", "Location", "Creation", "Control"), privacy, row_headers=True)
print("Data locations and consent points are explicit; no account is required.")
_html

Data locations and consent points are explicit; no account is required.


Data,Location,Creation,Control
Session state,Browser memory,Default,Ends with the session
Local slot,Browser storage,Explicit per-slot consent,Player can inspect and delete
Portable file,Player-selected location,Explicit export,Player controls transfer
Imported file,Temporary parser input,Explicit selection,No restore before validation


## Documentation publication contract

Notebook source and static HTML are released together. The source retains execution counts and outputs; the HTML requires no notebook runtime, remote font, script library, or network connection.

In [18]:
publication = [
    ("Canonical notebook", "notebooks/*.ipynb", "Executed cells and committed outputs"),
    ("Static edition", "public/notebooks/*.html", "Self-contained semantic document"),
    ("Download copy", "public/notebooks/*.ipynb", "Byte-identical to canonical source"),
    ("Manifest", "notebooks/artifact-manifest.json", "Size and SHA-256 for every published artifact"),
    ("Rebuild", "python3 scripts/docs/build_notebooks.py", "Standard library only"),
    ("Drift check", "python3 scripts/docs/build_notebooks.py --check", "Fails if committed output differs"),
]
_html = table_html("Notebook publication artifacts", ("Artifact", "Path", "Guarantee"), publication, row_headers=True)
print("PASS: notebook source, static edition, downloadable copy, and integrity manifest move together.")
_html

PASS: notebook source, static edition, downloadable copy, and integrity manifest move together.


Artifact,Path,Guarantee
Canonical notebook,notebooks/*.ipynb,Executed cells and committed outputs
Static edition,public/notebooks/*.html,Self-contained semantic document
Download copy,public/notebooks/*.ipynb,Byte-identical to canonical source
Manifest,notebooks/artifact-manifest.json,Size and SHA-256 for every published artifact
Rebuild,python3 scripts/docs/build_notebooks.py,Standard library only
Drift check,python3 scripts/docs/build_notebooks.py --check,Fails if committed output differs


## Release decision

This historical ledger does not promote the updated working tree. A release is ready only when generator coherence, reducer invariants, save validation, viewport contracts, accessibility disclosures, static rendering, documentation drift checks, browser completion, and distribution checks all pass for the same source binding.

In [19]:
release = [
    ("Generator coherence", "All constrained-night checks pass across the release seed sweep."),
    ("Reducer integrity", "Serial, reverse, interleaved, and delayed-entry stress paths remain valid."),
    ("Persistence", "Round-trip, migration, hostile-input, consent, and deletion tests pass."),
    ("Viewport", "Desktop and compact contracts retain one bounded game shell and owned scrolling."),
    ("Accessibility", "Keyboard, focus, reflow, naming, motion, contrast, and disclosure checks pass."),
    ("Rendered output", "Server response and static notebook documents contain required landmarks and metadata."),
    ("Documentation", "Notebook rebuild check reports no drift and manifest hashes match."),
]
rows = [("REQUIRED", label, evidence) for label, evidence in release]
_html = checklist_html("Release evidence gates", rows)
print(f"Release ledger defines {len(rows)} independent evidence gates; none is replaced by visual inspection alone.")
_html

Release ledger defines 7 independent evidence gates; none is replaced by visual inspection alone.


REQUIRED  Generator coherence  All constrained-night checks pass across the release seed sweep.    REQUIRED  Reducer integrity  Serial, reverse, interleaved, and delayed-entry stress paths remain valid.    REQUIRED  Persistence  Round-trip, migration, hostile-input, consent, and deletion tests pass.    REQUIRED  Viewport  Desktop and compact contracts retain one bounded game shell and owned scrolling.    REQUIRED  Accessibility  Keyboard, focus, reflow, naming, motion, contrast, and disclosure checks pass.    REQUIRED  Rendered output  Server response and static notebook documents contain required landmarks and metadata.    REQUIRED  Documentation  Notebook rebuild check reports no drift and manifest hashes match.